In [ ]:
# !pip install pandas rapidfuzz sentence-transformers faiss-cpu scikit-learn openai hdbscan
import json
import os
import re
import unicodedata

import hdbscan
import numpy as np
import openai
import pandas as pd
from rapidfuzz import fuzz, process
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

FUZZY_MATCH_THRESHOLD = 90  # score >= 90 → matched direto
FUZZY_LOWCONF_THRESHOLD = 70  # score entre 70-90 → passa para embedding
EMBEDDING_MATCH_THRESHOLD = 0.85  # score >= 0.85 → matched
EMBEDDING_LOWCONF_THRESHOLD = 0.75  # entre 0.75-0.85 → low_confidence, vai pro LLM
NO_MATCH_ABSOLUTE_THRESHOLD = 0.75  # abaixo disso → no_match direto

# multilingual-e5-large: bom balanço PT/EN, roda local sem custo de API
MODEL_NAME = "intfloat/multilingual-e5-large"

In [ ]:
def load_data(
    new_skills_path: str, taxonomy_path: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    new_skills = pd.read_csv(new_skills_path)
    taxonomy = pd.read_csv(taxonomy_path, sep=";")

    print(f"Skills brutas carregadas:    {len(new_skills)} registros")
    print(f"Skills na taxonomia:         {len(taxonomy)} registros")

    return new_skills, taxonomy


def normalize_text(text: str) -> str:
    """
    Normalização leve — preserva semântica mas remove ruído superficial.
    Não fazemos stemming para não perder informação.
    """
    if not isinstance(text, str):
        return ""
    # remove caracteres especiais desnecessários
    return re.sub(
        r"\s+",
        " ",
        re.sub(
            r"[^\w\s]",
            " ",
            "".join(
                c
                for c in unicodedata.normalize("NFKD", text.lower().strip())
                if not unicodedata.combining(c)
            ),
        ),
    ).strip()


def preprocess(
    new_skills: pd.DataFrame, taxonomy: pd.DataFrame
) -> tuple[dict[str, str], dict[str, str]]:
    new_skills = new_skills.copy()
    taxonomy = taxonomy.copy()

    new_skills["skill_normalized"] = new_skills["skill_raw"].apply(normalize_text)

    # Para taxonomia, concatenamos nome + descrição para enriquecer o sinal semântico
    # Isso é importante: "modelos de embedding" vai bater com
    # "Embeddings e Busca Vetorial" pelo contexto da descrição
    taxonomy["taxonomy_normalized"] = taxonomy["skill_name"].apply(normalize_text)
    taxonomy["taxonomy_text"] = taxonomy["skill_name"] + ". " + taxonomy["description"]

    return new_skills, taxonomy

In [ ]:
def fuzzy_match(
    skill_normalized: str, taxonomy: pd.DataFrame
) -> tuple[int | None, str | None, float]:
    """
    Retorna (taxonomy_id, skill_name, score) do melhor match fuzzy.
    Usa token_sort_ratio para ser robusto a ordem de palavras.
    Ex: "pessoas gestao" ainda bate com "gestao pessoas"
    """

    choices = dict(zip(taxonomy["taxonomy_normalized"], taxonomy["id"], strict=False))

    result = process.extractOne(
        skill_normalized, choices.keys(), scorer=fuzz.token_sort_ratio
    )
    if result is None:
        return None, None, 0.0

    best_match_text, score, _ = result
    taxonomy_id = choices[best_match_text]
    skill_name = taxonomy.loc[taxonomy["id"] == taxonomy_id, "skill_name"].values[0]

    return taxonomy_id, skill_name, score / 100  # normaliza para 0-1


def apply_fuzzy_layer(new_skills: pd.DataFrame, taxonomy: pd.DataFrame) -> pd.DataFrame:
    results = []

    for _, row in new_skills.iterrows():
        tax_id, skill_name, score = fuzzy_match(row["skill_normalized"], taxonomy)

        if score >= FUZZY_MATCH_THRESHOLD / 100:
            status = "matched"
        elif score >= FUZZY_LOWCONF_THRESHOLD / 100:
            status = "needs_embedding"  # passa para próxima camada
        else:
            status = "needs_embedding"  # score muito baixo, mas tenta embedding

        results.append(
            {
                "new_skill_id": row["id"],
                "skill_raw": row["skill_raw"],
                "skill_normalized": row["skill_normalized"],
                "taxonomy_id": tax_id if score >= FUZZY_MATCH_THRESHOLD / 100 else None,
                "skill_name": skill_name
                if score >= FUZZY_MATCH_THRESHOLD / 100
                else None,
                "confidence_score": round(score, 4),
                "match_status": status,
                "matched_by": "fuzzy" if score >= FUZZY_MATCH_THRESHOLD / 100 else None,
            }
        )
    return pd.DataFrame(results)

In [ ]:
def build_taxonomy_embeddings(
    taxonomy: pd.DataFrame, model: SentenceTransformer
) -> np.ndarray:
    """
    Gera embeddings para toda a taxonomia uma única vez.
    Embedamos 'skill_name. description' para maximizar sinal semântico.
    """
    print("Gerando embeddings da taxonomia...")
    # e5 precisa do prefixo "passage:" para textos de referência
    return model.encode(  # normaliza para que cosine = dot product
        [f"passage: {t}" for t in taxonomy["taxonomy_text"].tolist()],
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )


def embedding_match(
    skill_text: str,
    taxonomy: pd.DataFrame,
    taxonomy_embeddings: np.ndarray,
    model: SentenceTransformer,
) -> tuple[int | None, str | None, float]:
    """
    Retorna (taxonomy_id, skill_name, score) do match por embedding.
    """
    # e5 precisa do prefixo "query:" para textos de busca
    # similaridade de cosseno contra toda a taxonomia
    similarities = cosine_similarity(
        model.encode([f"query: {skill_text}"], normalize_embeddings=True),
        taxonomy_embeddings,
    )[0]
    best_idx = int(np.argmax(similarities))

    return (
        taxonomy.iloc[best_idx]["id"],
        taxonomy.iloc[best_idx]["skill_name"],
        float(similarities[best_idx]),
    )


def apply_embedding_layer(
    df: pd.DataFrame,
    taxonomy: pd.DataFrame,
    taxonomy_embeddings: np.ndarray,
    model: SentenceTransformer,
) -> pd.DataFrame:
    df = df.copy()
    pending_mask = df["match_status"] == "needs_embedding"
    pending_df = df[pending_mask]
    print(f"\nSkills para embedding: {len(pending_df)}")

    if len(pending_df) == 0:
        return df

    texts = [f"query: {t}" for t in pending_df["skill_raw"].tolist()]
    skill_embeddings = model.encode(
        texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True
    )
    # cosine_similarity(M skills, N taxonomy) uma única vez → matriz (M x N)
    similarity_matrix = cosine_similarity(skill_embeddings, taxonomy_embeddings)

    best_idxs = similarity_matrix.argmax(axis=1)  # índice do melhor match por skill
    best_scores = similarity_matrix.max(axis=1)  # score do melhor match por skill

    taxonomy_ids = taxonomy.iloc[best_idxs]["id"].values
    skill_names = taxonomy.iloc[best_idxs]["skill_name"].values

    conditions = [  # classifica cada skill com base nos thresholds
        best_scores < NO_MATCH_ABSOLUTE_THRESHOLD,
        best_scores >= EMBEDDING_MATCH_THRESHOLD,
    ]
    # entre NO_MATCH e EMBEDDING_MATCH
    statuses = np.select(conditions, ["no_match", "matched"], default="needs_llm")
    matched_by = np.where(statuses == "matched", "embedding", None)

    no_match_mask_local = statuses == "no_match"
    taxonomy_ids[no_match_mask_local] = None
    skill_names[no_match_mask_local] = None

    df.loc[pending_mask, "taxonomy_id"] = taxonomy_ids
    df.loc[pending_mask, "skill_name"] = skill_names
    df.loc[pending_mask, "confidence_score"] = best_scores.round(4)
    df.loc[pending_mask, "match_status"] = statuses
    df.loc[pending_mask, "matched_by"] = matched_by

    return df

In [ ]:
def llm_judge(
    skill_raw: str,
    candidate_name: str,
    candidate_description: str,
    client: openai.OpenAI,
) -> tuple[bool, str]:
    """
    Usa o LLM como árbitro para casos ambíguos.
    Retorna (is_match, justificativa). Peço por favor pra prevenir a Skynet
    """

    prompt = f"""Você é um especialista em classificação de habilidades profissionais.

Avalie se a skill abaixo representa a mesma competência que a skill da taxonomia.

Skill registrada: "{skill_raw}"

Skill da taxonomia: "{candidate_name}"
Descrição: "{candidate_description}"

Responda APENAS no formato JSON por favor:
{{
  "match": true ou false,
  "justificativa": "explicação em uma frase"
}}

Considere match=true se a skill registrada é um sinônimo, variação de idioma, 
ou subconjunto claro da skill da taxonomia.
Considere match=false se há dúvida razoável ou são competências distintas."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",  # barato e suficiente para classificação binária
        messages=[{"role": "user", "content": prompt}],
        temperature=0,  # queremos resposta determinística
        response_format={"type": "json_object"},
    )

    result = json.loads(response.choices[0].message.content)
    return result["match"], result["justificativa"]


def apply_llm_layer(
    df: pd.DataFrame, taxonomy: pd.DataFrame, client: openai.OpenAI
) -> pd.DataFrame:
    df = df.copy()
    pending_mask = df["match_status"] == "needs_llm"

    print(f"\nSkills para LLM judge: {pending_mask.sum()}")

    for idx, row in df[pending_mask].iterrows():
        tax_row = taxonomy[taxonomy["id"] == row["taxonomy_id"]].iloc[0]
        description = tax_row["description"]

        is_match, justificativa = llm_judge(
            row["skill_raw"], row["skill_name"], description, client
        )

        if is_match:
            df.at[idx, "match_status"] = "matched"
            df.at[idx, "matched_by"] = "llm"
        else:
            df.at[idx, "taxonomy_id"] = None
            df.at[idx, "skill_name"] = None
            df.at[idx, "match_status"] = "no_match"
            df.at[idx, "matched_by"] = None

        # guarda justificativa para auditoria
        df.at[idx, "llm_justificativa"] = justificativa

    return df

In [ ]:
def cluster_no_matches(
    df: pd.DataFrame, taxonomy_embeddings: np.ndarray, model: SentenceTransformer
) -> pd.DataFrame:
    """
    Agrupa skills sem match entre si para identificar
    candidatas a novas entradas na taxonomia.
    """
    no_match_mask = df["match_status"] == "no_match"
    no_match_df = df[no_match_mask].copy()

    if len(no_match_df) < 2:
        df.loc[no_match_mask, "cluster_id"] = -1
        return df

    print(f"\nClusterizando {len(no_match_df)} skills sem match...")

    texts = [f"query: {t}" for t in no_match_df["skill_raw"].tolist()]
    embeddings = model.encode(texts, normalize_embeddings=True)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=2, metric="euclidean", cluster_selection_method="eom"
    )
    labels = clusterer.fit_predict(embeddings)

    df.loc[no_match_mask, "cluster_id"] = labels

    # -1 = ruído (skill única, sem par similar)
    print(f"Clusters encontrados: {len(set(labels)) - (1 if -1 in labels else 0)}")
    print(f"Skills isoladas (ruído): {(labels == -1).sum()}")

    return df

In [ ]:
def run_pipeline(
    new_skills_path: str,
    taxonomy_path: str,
    openai_api_key: str | None = None,
    output_path: str = "output/skill_mapping.csv",
) -> pd.DataFrame:

    new_skills, taxonomy = load_data(new_skills_path, taxonomy_path)
    new_skills, taxonomy = preprocess(new_skills, taxonomy)

    print("\n[1/4] Fuzzy matching...")
    results = apply_fuzzy_layer(new_skills, taxonomy)

    fuzzy_matched = (results["match_status"] == "matched").sum()
    print(f"Resolvidos por fuzzy: {fuzzy_matched}/{len(results)}")

    print("\n[2/4] Carregando modelo de embedding...")
    model = SentenceTransformer(MODEL_NAME)
    taxonomy_embeddings = build_taxonomy_embeddings(taxonomy, model)

    results = apply_embedding_layer(results, taxonomy, taxonomy_embeddings, model)

    embedding_matched = (
        (results["match_status"] == "matched") & (results["matched_by"] == "embedding")
    ).sum()
    print(f"Resolvidos por embedding: {embedding_matched}")

    if openai_api_key and (results["match_status"] == "needs_llm").sum() > 0:
        print("\n[3/4] LLM judge para casos ambíguos...")
        client = openai.OpenAI(api_key=openai_api_key)
        results = apply_llm_layer(results, taxonomy, client)

        llm_matched = (
            (results["match_status"] == "matched") & (results["matched_by"] == "llm")
        ).sum()
        print(f"Resolvidos por LLM: {llm_matched}")
    else:
        # sem LLM, needs_llm vira low_confidence no output
        results.loc[results["match_status"] == "needs_llm", "match_status"] = (
            "low_confidence"
        )
        print("\n[3/4] LLM pulado (sem API key ou sem casos ambíguos)")

    print("\n[4/4] Clusterizando skills sem match...")
    results = cluster_no_matches(results, taxonomy_embeddings, model)

    output_cols = [
        "new_skill_id",
        "skill_raw",
        "taxonomy_id",
        "skill_name",
        "confidence_score",
        "match_status",
        "matched_by",
        "cluster_id",
    ]
    if "llm_justificativa" in results.columns:
        output_cols.append("llm_justificativa")

    final = results[output_cols].copy()

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    final.to_csv(output_path, index=False)
    print(f"\nOutput salvo em: {output_path}")

    return final

In [ ]:
resultado = run_pipeline(
    new_skills_path="../data/raw/new_skills.csv",
    taxonomy_path="../data/raw/skill_taxonomy.csv",
    openai_api_key=os.getenv("OPENAI_API_KEY"),  # opcional
    output_path="../data/processed/skill_mapping.csv",
)
print("\n--- Resumo ---")
print(resultado["match_status"].value_counts())
print("\n--- Amostra ---")
print(resultado.to_string(index=False))

In [ ]:
def evaluate_quality(results: pd.DataFrame) -> dict:
    """
    Métricas de qualidade sem precisar de ground truth.
    """
    total = len(results)

    metrics = {
        # cobertura: quantas foram mapeadas com algum nível de confiança
        "coverage_rate": (results["match_status"] == "matched").sum() / total,
        # taxa de no_match: se muito alta, taxonomia pode estar desatualizada
        "no_match_rate": (results["match_status"] == "no_match").sum() / total,
        # taxa de low_confidence: indica zona de incerteza do modelo
        "low_confidence_rate": (results["match_status"] == "low_confidence").sum()
        / total,
        # distribuição dos scores para matched
        "mean_confidence_matched": results[results["match_status"] == "matched"][
            "confidence_score"
        ].mean(),
        # scores abaixo de 0.75 mesmo nos matched = sinal de alerta
        "pct_matched_below_075": (
            results[results["match_status"] == "matched"]["confidence_score"] < 0.75
        ).mean(),
        # quantas foram escaladas para LLM (custo)
        "llm_usage_rate": (results["matched_by"] == "llm").sum() / total,
        # breakdown por método
        "matched_by_fuzzy": (results["matched_by"] == "fuzzy").sum(),
        "matched_by_embedding": (results["matched_by"] == "embedding").sum(),
        "matched_by_llm": (results["matched_by"] == "llm").sum(),
    }
    print("\n=== MÉTRICAS DE QUALIDADE ===")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.2%}" if "rate" in k or "pct" in k else f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

    return metrics


evaluate_quality(resultado)